# 05 — Quantum-Hybrid Comparison

The project ships several optimisation paths — classical, quantum-inspired,
and the hybrid pipeline. This notebook runs them on the same universe and
shows where each one shines (and where it doesn't).

**Objectives compared:**

| Family | Objective | What it does |
|---|---|---|
| Baseline | `equal_weight` | 1/N — surprisingly hard to beat |
| Classical | `markowitz` | max Sharpe via SLSQP |
| Classical | `min_variance` | global minimum variance |
| Classical | `hrp` | Hierarchical Risk Parity |
| Convex | `mean_cvar` | scenario-based tail-risk LP |
| Quantum-inspired | `qubo_sa` | QUBO + simulated annealing |
| Hybrid | `hybrid` | IC screen → QUBO → Markowitz |

## 1. Universe + scenarios

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from benchmarks.base import generate_synthetic_dataset
from services.scenario_generation import ScenarioConfig, generate_scenarios
from core.portfolio_optimizer import run_optimization

ds = generate_synthetic_dataset(n_assets=10, n_history=504, seed=42)
tickers = [f"A{i:02d}" for i in range(ds.n_assets)]
scenarios = generate_scenarios(
    ds.daily_returns,
    ScenarioConfig(method="block", n_scenarios=5000, seed=42),
)

## 2. Run every objective

We standardise on `weight_max=0.30` so the cardinality / concentration
rules are comparable. The QUBO-SA path uses `K=5` (select 5 of 10 assets).

In [ ]:
common = dict(returns=ds.mu, covariance=ds.Sigma, asset_names=tickers,
              weight_min=0.0, weight_max=0.30, seed=42)

objectives = [
    {"objective": "equal_weight"},
    {"objective": "markowitz"},
    {"objective": "min_variance"},
    {"objective": "hrp"},
    {"objective": "mean_cvar", "scenarios": scenarios, "risk_aversion": 1.0},
    {"objective": "qubo_sa", "K": 5},
    {"objective": "hybrid", "K_screen": 8, "K_select": 5},
]

runs = {}
for spec in objectives:
    name = spec["objective"]
    runs[name] = run_optimization(**common, **spec)
    print(f"{name:>13s}  sharpe={runs[name].sharpe_ratio:.3f}")

## 3. Realised tail risk on the scenario panel

In [ ]:
def realised_cvar(weights, scenarios=scenarios, alpha=0.05):
    losses = -(scenarios @ weights)
    var = float(np.quantile(losses, 1 - alpha))
    tail = losses[losses >= var]
    return float(var), float(tail.mean()) if tail.size > 0 else var

header = ["objective", "return", "vol", "sharpe", "var95", "cvar95", "n_active"]
print("  ".join(f"{h:>11}" for h in header))
for name, r in runs.items():
    var, cvar = realised_cvar(r.weights)
    cells = [name,
             f"{r.expected_return:.4f}",
             f"{r.volatility:.4f}",
             f"{r.sharpe_ratio:.4f}",
             f"{var:.4f}", f"{cvar:.4f}",
             f"{r.n_active}"]
    print("  ".join(f"{c:>11}" for c in cells))

## 4. Weight allocations

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(tickers))
n = len(runs)
width = 0.8 / n
for i, (name, r) in enumerate(runs.items()):
    ax.bar(x + (i - n / 2) * width + width / 2, r.weights, width=width, label=name)
ax.set_xticks(x)
ax.set_xticklabels(tickers)
ax.set_ylabel("weight")
ax.set_title("Weight allocation by optimisation objective")
ax.legend(fontsize=8, ncol=4)
plt.tight_layout()
plt.show()

## 5. When does the quantum/hybrid path help?

From the table above, three patterns usually emerge:

1. **`equal_weight`** is the baseline — beat it on a risk-adjusted basis
   before celebrating any other objective.
2. **`markowitz` and `mean_cvar`** give the highest in-sample Sharpe
   under our bounds because they have a continuous weight space.
3. **`qubo_sa` / `hybrid`** shine when there are *real* discrete
   constraints — fixed cardinality, must-hold lists, sector caps with
   integer counts. On a simple universe like this, they trade some
   Sharpe for the explicit selection structure.

Replace `ds` with a larger universe and add a `cardinality` constraint
(via `PortfolioConstraints`) to see the hybrid stack pull ahead.

**Where to go next:**
- [docs/MEAN_CVAR.md](../../../docs/MEAN_CVAR.md)
- [docs/CONSTRAINT_ENGINE.md](../../../docs/CONSTRAINT_ENGINE.md)
- [docs/SOLVER_BACKENDS.md](../../../docs/SOLVER_BACKENDS.md)